In [ ]:
import json
import os
import re
import time
import collections
import ipaddress
import numpy as np
import pandas as pd
import requests
import matplotlib.patches as mpatches
import joblib
import matplotlib.pyplot as plt
from huggingface_hub import hf_hub_download
from pathlib import Path
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestRegressor
from sklearn.inspection import permutation_importance as sk_permutation_importance
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from IPython.display import display
from shared_variables import (
    IP_RANGES_DIR,
    PROVIDERS,
    HF_REPO_ID,
    HF_REPO_TYPE,
    CONSENSUS_DATA_FILENAME,
    EXECUTION_DATA_FILENAME,
)

## Download crawl data, extract ips, take out non-mainnet nodes

In [ ]:
def read_jsonl(path):
    records = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                records.append(json.loads(line))
    return records

def read_jsonl_from_hf(filename):
    path = hf_hub_download(
        repo_id=HF_REPO_ID,
        repo_type=HF_REPO_TYPE,
        filename=filename,
    )
    return read_jsonl(path)

cl_records = read_jsonl_from_hf(CONSENSUS_DATA_FILENAME)
el_records = read_jsonl_from_hf(EXECUTION_DATA_FILENAME)

In [ ]:
cl_top = pd.DataFrame([{k: v for k, v in r.items() if k != "peer_properties"} for r in cl_records])
cl_props = pd.json_normalize([r["peer_properties"] for r in cl_records])
df_consensus_visits = pd.concat([cl_top, cl_props], axis=1)

el_top = pd.DataFrame([{k: v for k, v in r.items() if k != "peer_properties"} for r in el_records])
el_props = pd.json_normalize([r["peer_properties"] for r in el_records])
df_execution_visits = pd.concat([el_top, el_props], axis=1)

In [ ]:
df_el = df_execution_visits.copy()
df_cl = df_consensus_visits.copy()

In [ ]:
def extract_ip_from_connect_maddr(maddr):
    if not maddr:
        return pd.NA
    parts = str(maddr).split("/")
    if len(parts) >= 3 and parts[1] == "ip4":
        return parts[2]
    return pd.NA

df_el = df_el.copy()
df_el["ip"] = df_el["connect_maddr"].apply(extract_ip_from_connect_maddr)
df_el = df_el[df_el["ip"].notna()].copy()

df_cl = df_cl.copy()
df_cl["ip"] = df_cl["connect_maddr"].apply(extract_ip_from_connect_maddr)
df_cl = df_cl[df_cl["ip"].notna()].copy()

print(f"EL rows with IP: {len(df_el)}")
print(f"CL rows with IP: {len(df_cl)}")

df_cl_pair = df_cl.copy()
df_el_pair = df_el.copy()

df_cl_pair["ip_occurrence"] = df_cl_pair.groupby("ip").cumcount()
df_el_pair["ip_occurrence"] = df_el_pair.groupby("ip").cumcount()

df_cl_renamed = df_cl_pair.rename(columns=lambda c: c + "_cl" if c not in {"ip", "ip_occurrence"} else c)
df_el_renamed = df_el_pair.rename(columns=lambda c: c + "_el" if c not in {"ip", "ip_occurrence"} else c)

df_merged = df_cl_renamed.merge(df_el_renamed, on=["ip", "ip_occurrence"], how="inner")

print(df_merged.shape)

In [ ]:
df_merged = df_merged[
    (df_merged["fork_digest_cl"].notna())
].copy()

print(df_merged.shape)

## Extract clients, syncnets, hw_arch, os_token, cloud_providers, attnets cleaning

In [ ]:
CL_CLIENT_NAMES = ["lighthouse", "prysm", "teku", "nimbus", "lodestar", "grandine", "caplin"]
EL_CLIENT_NAMES = ["geth", "nethermind", "besu", "erigon", "reth", "nimbus", "ethrex"]

In [ ]:
_CL_PATTERN_SOURCES = {
    "lighthouse": re.compile(r"^lighthouse", re.IGNORECASE),
    "prysm":      re.compile(r"^prysm",      re.IGNORECASE),
    "teku":       re.compile(r"^teku",        re.IGNORECASE),
    "nimbus":     re.compile(r"^nimbus",      re.IGNORECASE),
    "lodestar":   re.compile(r"^lodestar",    re.IGNORECASE),
    "grandine":   re.compile(r"^grandine",    re.IGNORECASE),
    "caplin":     re.compile(r"caplin",      re.IGNORECASE),
}

_EL_PATTERN_SOURCES = {
    "geth":       re.compile(r"^geth",      re.IGNORECASE),
    "nethermind": re.compile(r"^nethermind", re.IGNORECASE),
    "besu":       re.compile(r"^besu",      re.IGNORECASE),
    "erigon":     re.compile(r"^erigon",     re.IGNORECASE),
    "reth":       re.compile(r"^reth",      re.IGNORECASE),
    "nimbus":     re.compile(r"^nimbus",      re.IGNORECASE),
    "ethrex":     re.compile(r"^ethrex",      re.IGNORECASE),
}

_CL_PATTERNS = [(name, _CL_PATTERN_SOURCES[name]) for name in CL_CLIENT_NAMES]
_EL_PATTERNS = [(name, _EL_PATTERN_SOURCES[name]) for name in EL_CLIENT_NAMES]

def parse_client_name(agent, patterns):
    return next((name for name, pat in patterns if pat.search(agent)), pd.NA)

df_merged["consensus_client"] = df_merged["agent_version_cl"].apply(lambda v: parse_client_name(v, _CL_PATTERNS))
df_merged["execution_client"] = df_merged["agent_version_el"].apply(lambda v: parse_client_name(v, _EL_PATTERNS))

In [ ]:
def _syncnets_count(s):
    try:
        return bin(int(s, 16)).count("1")
    except Exception:
        return 0

df_merged["syncnets_num"] = df_merged["syncnets_cl"].apply(lambda s: _syncnets_count(str(s)))

print(df_merged.shape)

In [ ]:
_ARM_TOKENS = {"aarch64", "aarch_64", "arm64"}
_X86_TOKENS = {"x86_64", "amd64", "linux-x64", "windows-x64", "linux-386", "x86_64-unknown-linux-gnu"}


def parse_hw_arch(agent):
    low = agent.lower()
    is_arm = any(tok in low for tok in _ARM_TOKENS)
    is_x86 = any(tok in low for tok in _X86_TOKENS)
    if is_arm and is_x86:
        return pd.NA
    if is_arm:
        return "ARM"
    if is_x86:
        return "x86"
    return pd.NA

def resolve_arch_and_os(c, e):
    if pd.isna(c):
        return e
    if pd.isna(e):
        return c
    if c == e:
        return c
    return pd.NA

cl_arches = df_merged["agent_version_cl"].apply(parse_hw_arch)
el_arches = df_merged["agent_version_el"].apply(parse_hw_arch)
df_merged["hw_arch"] = [resolve_arch_and_os(c, e) for c, e in zip(cl_arches, el_arches)]

print(df_merged.shape)

In [ ]:
_OS_TOKEN_GROUPS = {
    "linux":   {"linux"},
    "macos":   {"macos", "darwin", "osx"},
    "windows": {"windows"},
}

def parse_os_token(agent):
    low = agent.lower()
    return next((os_name for os_name, toks in _OS_TOKEN_GROUPS.items() if any(tok in low for tok in toks)), pd.NA)

cl_os_tokens = df_merged["agent_version_cl"].apply(parse_os_token)
el_os_tokens = df_merged["agent_version_el"].apply(parse_os_token)
df_merged["os_token"] = [resolve_arch_and_os(c, e) for c, e in zip(cl_os_tokens, el_os_tokens)]

print(df_merged.shape)

In [ ]:
def load_ip_indices():
    indices = []
    for provider in PROVIDERS:
        path = IP_RANGES_DIR / f"{provider}.json"
        data = json.loads(path.read_text())
        nets = [ipaddress.ip_network(p, strict=False) for p in data["prefixes"]]
        idx = collections.defaultdict(list)
        for net in nets:
            start = int(net.network_address) >> 24
            end = int(net.broadcast_address) >> 24
            for bucket in range(start, end + 1):
                idx[bucket].append(net)
        indices.append((provider, dict(idx)))
    return indices

def classify_ip(ip_str, indices):
    addr = ipaddress.ip_address(ip_str)
    for provider, idx in indices:
        if any(addr in net for net in idx.get(int(addr) >> 24, [])):
            return provider
    return pd.NA

indices = load_ip_indices()
df_merged["cloud_provider"] = df_merged["ip"].apply(lambda ip: classify_ip(ip, indices))

In [ ]:
df_merged["attnets_num_cl"] = pd.to_numeric(df_merged["attnets_num_cl"], errors="coerce").fillna(0).astype(int)

In [ ]:
import ast

_BACKBONE_SUBNETS_PER_NODE = 2

def _count_attestation_gossip_topics(topics_str):
    try:
        topics = topics_str if isinstance(topics_str, list) else ast.literal_eval(str(topics_str))
        return sum(1 for t in topics if "beacon_attestation_" in t)
    except Exception:
        return 0

df_merged["gossipsub_att_count"] = df_merged["gossipsub_topics_cl"].apply(
    _count_attestation_gossip_topics
)

## Power values nodes that match requirements

In [ ]:
client_fields_present = (
    df_merged["consensus_client"].notna()
    & df_merged["execution_client"].notna()
    & df_merged["hw_arch"].notna()
    & df_merged["os_token"].notna()
)
caplin_invalid_mask = (df_merged["consensus_client"] == "caplin") & (df_merged["execution_client"] != "erigon")

df = df_merged[client_fields_present & ~caplin_invalid_mask].copy()

In [ ]:
_KEEP_FOR_INFERENCE = {
    "consensus_client": "consensus_client",
    "attnets_num_cl": "attnets_num",
    "syncnets_num": "syncnets_num",
    "gossipsub_att_count": "gossipsub_att_count",
    "execution_client": "execution_client",
    "cloud_provider": "cloud_provider",
    "os_token": "os_token",
    "hw_arch": "hw_arch",
}

df = df[[c for c in _KEEP_FOR_INFERENCE]]
df = df.rename(columns=_KEEP_FOR_INFERENCE)

print(df.shape)

In [ ]:
df["is_validator"] = (
    (df["attnets_num"] > _BACKBONE_SUBNETS_PER_NODE)
    | (df["gossipsub_att_count"] > _BACKBONE_SUBNETS_PER_NODE)
    | (df["syncnets_num"] > 0)
)

print(df.shape)

### Bare-Metal, CCRI

In [ ]:
_TIER_WEIGHTS = {5: 0.75, 6: 0.25}
_IDLE = {5: 25.04, 6: 78.17}

_CL_MARGINAL = {
    "lighthouse": {5: 3.14,  6: 18.84},
    "prysm":      {5: 2.87,  6: 24.33},
    "teku":       {5: 3.32,  6: 27.46},
    "nimbus":     {5: 2.08,  6: 17.11},
    "lodestar":   {5: 3.89,  6: 33.55},
}

_EL_MARGINAL = {
    "geth":   {5: 9.70,  6: 47.70},
    "erigon": {5: 17.59, 6: 44.62},
    "besu":   {5: 31.02, 6: 75.04},
}

def _weighted(per_tier):
    return sum(_TIER_WEIGHTS[t] * v for t, v in per_tier.items())

WEIGHTED_IDLE_W = _weighted(_IDLE)
COMBINED_ADJUSTMENT_FACTOR = 0.91

CCRI_CL_MARGINAL_W = {k: _weighted(v) for k, v in _CL_MARGINAL.items()}
CCRI_EL_MARGINAL_W = {k: _weighted(v) for k, v in _EL_MARGINAL.items()}

PROXY_CL_MARGINAL_W = {
    "grandine": CCRI_CL_MARGINAL_W["nimbus"],
    "caplin":   CCRI_CL_MARGINAL_W["nimbus"],
}

PROXY_EL_MARGINAL_W = {
    "nethermind": CCRI_EL_MARGINAL_W["geth"],
    "reth":       CCRI_EL_MARGINAL_W["erigon"],
    "nimbus":     CCRI_EL_MARGINAL_W["geth"],
    "ethrex":     CCRI_EL_MARGINAL_W["geth"],
}

_ALL_CL_MARGINAL_W = {**CCRI_CL_MARGINAL_W, **PROXY_CL_MARGINAL_W}
_ALL_EL_MARGINAL_W = {**CCRI_EL_MARGINAL_W, **PROXY_EL_MARGINAL_W}

if set(_ALL_CL_MARGINAL_W) != set(CL_CLIENT_NAMES):
    raise ValueError(f"CL marginal power coverage mismatch: {set(CL_CLIENT_NAMES) ^ set(_ALL_CL_MARGINAL_W)}")

if set(_ALL_EL_MARGINAL_W) != set(EL_CLIENT_NAMES):
    raise ValueError(f"EL marginal power coverage mismatch: {set(EL_CLIENT_NAMES) ^ set(_ALL_EL_MARGINAL_W)}")

ARM_LINUX_NODE_W_WEB3PI = 10.0

_MACOS_IDLE_SAMPLES_W = [6.8, 7.0]
ARM_MACOS_IDLE_W = sum(_MACOS_IDLE_SAMPLES_W) / len(_MACOS_IDLE_SAMPLES_W)
x86_MACOS_IDLE_W = 19.9

In [ ]:
df["power_cl_marginal_w"] = df["consensus_client"].map(_ALL_CL_MARGINAL_W)
df["power_el_marginal_w"] = df["execution_client"].map(_ALL_EL_MARGINAL_W)

arm_non_cloud = (df["hw_arch"] == "ARM") & df["cloud_provider"].isna()
x86_non_cloud = (df["hw_arch"] == "x86") & df["cloud_provider"].isna()

os_linux = df["os_token"].isin(_OS_TOKEN_GROUPS["linux"])
os_windows = df["os_token"].isin(_OS_TOKEN_GROUPS["windows"])
os_macos = df["os_token"].isin(_OS_TOKEN_GROUPS["macos"])

arm_linux_mask = arm_non_cloud & os_linux
arm_macos_mask = arm_non_cloud & os_macos
arm_windows_mask = arm_non_cloud & os_windows

x86_linux_mask = x86_non_cloud & os_linux 
x86_macos_mask = x86_non_cloud & os_macos
x86_windows_mask = x86_non_cloud & os_windows

df.loc[arm_linux_mask, "power_node_w"] = ARM_LINUX_NODE_W_WEB3PI

df.loc[x86_linux_mask | x86_windows_mask, "power_node_w"] = (
    (df.loc[x86_linux_mask | x86_windows_mask, "power_cl_marginal_w"] + df.loc[x86_linux_mask | x86_windows_mask, "power_el_marginal_w"])
    * COMBINED_ADJUSTMENT_FACTOR
    + WEIGHTED_IDLE_W
)

df.loc[arm_macos_mask, "power_node_w"] = (
    (df.loc[arm_macos_mask, "power_cl_marginal_w"] + df.loc[arm_macos_mask, "power_el_marginal_w"])
    * COMBINED_ADJUSTMENT_FACTOR
    + ARM_MACOS_IDLE_W
)

df.loc[x86_macos_mask, "power_node_w"] = (
    (df.loc[x86_macos_mask, "power_cl_marginal_w"] + df.loc[x86_macos_mask, "power_el_marginal_w"])
    * COMBINED_ADJUSTMENT_FACTOR
    + x86_MACOS_IDLE_W
)

print(df.shape)

In [ ]:
df = df.loc[~arm_windows_mask].copy()

print(df.shape)

### Cloud, AWS, CCF, Pankovska

In [ ]:
SSD_OVERHEAD_W_PANKOVSKA = 5.0
CLOUD_PUE_PANKOVSKA = 1.2

CLOUD_PUE_CCF = 1.185

TEADS_MEMORY_W_PER_GB = 0.60
CCF_SSD_W_PER_TB = 1.2
NODE_SSD_NON_VALIDATOR_TB = 2.0
NODE_SSD_VALIDATOR_TB = 4.0

NODE_VCPU_MIN_NON_VALIDATOR = 4
NODE_VCPU_MIN_VALIDATOR = 8
NODE_RAM_NON_VALIDATOR_GB = 32
NODE_RAM_VALIDATOR_GB = 64

_M6I_INSTANCES = {
    "m6i.2xlarge": {"vcpu": 8,  "ram_gb": 32,  "arch": "x86", "cpu": "Xeon Platinum 8375C", "pkg_w_100": 38.20, "ram_w_100": 19.20, "delta": 7.5,  "pct100": 64.90},
    "m6i.4xlarge": {"vcpu": 16, "ram_gb": 64,  "arch": "x86", "cpu": "Xeon Platinum 8375C", "pkg_w_100": 76.39, "ram_w_100": 38.40, "delta": 15.0, "pct100": 129.79},
}

_R6I_INSTANCES = {
    "r6i.2xlarge": {"vcpu": 8,  "ram_gb": 64,  "arch": "x86", "cpu": "Xeon Platinum 8375C", "pkg_w_100": _M6I_INSTANCES["m6i.2xlarge"]["pkg_w_100"], "ram_w_100": 2 * _M6I_INSTANCES["m6i.2xlarge"]["ram_w_100"], "delta": _M6I_INSTANCES["m6i.2xlarge"]["delta"], "pct100": _M6I_INSTANCES["m6i.2xlarge"]["pkg_w_100"] + 2 * _M6I_INSTANCES["m6i.2xlarge"]["ram_w_100"] + _M6I_INSTANCES["m6i.2xlarge"]["delta"]},
    "r6i.4xlarge": {"vcpu": 16, "ram_gb": 128, "arch": "x86", "cpu": "Xeon Platinum 8375C", "pkg_w_100": _M6I_INSTANCES["m6i.4xlarge"]["pkg_w_100"], "ram_w_100": 2 * _M6I_INSTANCES["m6i.4xlarge"]["ram_w_100"], "delta": _M6I_INSTANCES["m6i.4xlarge"]["delta"], "pct100": _M6I_INSTANCES["m6i.4xlarge"]["pkg_w_100"] + 2 * _M6I_INSTANCES["m6i.4xlarge"]["ram_w_100"] + _M6I_INSTANCES["m6i.4xlarge"]["delta"]},
}

_R6G_INSTANCES = {
    "r6g.2xlarge": {"vcpu": 8,  "ram_gb": 64,  "arch": "ARM", "pkg_w_100": 19.10, "ram_w_100": 38.40, "delta": 3.75, "pct100": 61.20},
    "r6g.4xlarge": {"vcpu": 16, "ram_gb": 128, "arch": "ARM", "pkg_w_100": 38.20, "ram_w_100": 76.80, "delta": 7.50, "pct100": 122.50},
}

_GRAVITON3_CHIP_W_100 = 100.0
_GRAVITON3_TOTAL_VCPUS = 64

_M6G_INSTANCES = {
    "m6g.2xlarge": {"vcpu": 8,  "ram_gb": 32, "arch": "ARM", "pkg_w_100": 19.10, "ram_w_100": 19.20, "delta": 3.8},
    "m6g.4xlarge": {"vcpu": 16, "ram_gb": 64, "arch": "ARM", "pkg_w_100": 38.20, "ram_w_100": 38.40, "delta": 7.5},
}

_M7G_INSTANCES_BASE = {
    "m7g.2xlarge": _M6G_INSTANCES["m6g.2xlarge"],
    "m7g.4xlarge": _M6G_INSTANCES["m6g.4xlarge"],
}

_M7G_INSTANCES_PKG = {
    name: {
        **v,
        "pkg_w_100": round(_GRAVITON3_CHIP_W_100 * (v["vcpu"] / _GRAVITON3_TOTAL_VCPUS), 2),
    }
    for name, v in _M7G_INSTANCES_BASE.items()
}

_M7G_INSTANCES = {
    name: {
        "vcpu":      v["vcpu"],
        "ram_gb":    v["ram_gb"],
        "arch":      v["arch"],
        "pkg_w_100": v["pkg_w_100"],
        "ram_w_100": v["ram_w_100"],
        "delta":     v["delta"],
        "pct100":    v["pkg_w_100"] + v["ram_w_100"] + v["delta"],
    }
    for name, v in _M7G_INSTANCES_PKG.items()
}

AWS_EC2_INSTANCE_POWER_W = {
    name: {"vcpu": v["vcpu"], "ram_gb": v["ram_gb"], "arch": v["arch"], "pct100": v["pct100"]}
    for name, v in {**_M6I_INSTANCES, **_R6I_INSTANCES, **_R6G_INSTANCES, **_M7G_INSTANCES}.items()
}

CCF_VCPU_MAX_W = {
    "aws":   3.50,
    "gcp":   4.26,
    "azure": 3.76,
}

_CCF_MICROARCH_MAX_W = {
    "EPYC 1st Gen":    2.6042,
    "EPYC 2nd Gen":    1.6930,
    "EPYC 3rd Gen":    1.9573,
    "EPYC 4th Gen":    2.2822,
    "EPYC 5th Gen":    8.9614,
    "Cascade Lake":    4.0632,
    "Ice Lake":        3.7582,
    "Sapphire Rapids": 4.1605,
    "Skylake":         4.1042,
}

_PROVIDER_MICROARCHS = {
    "hetzner":      ["EPYC 2nd Gen", "EPYC 3rd Gen", "EPYC 4th Gen"],
    "ovh":          ["EPYC 3rd Gen", "EPYC 4th Gen", "Cascade Lake", "Sapphire Rapids"],
    "contabo":      ["EPYC 2nd Gen", "EPYC 4th Gen", "EPYC 5th Gen"],
    "netcup":       ["EPYC 2nd Gen", "EPYC 4th Gen", "EPYC 5th Gen"],
    "digitalocean": ["Skylake", "Cascade Lake", "Ice Lake", "Sapphire Rapids", "EPYC 2nd Gen", "EPYC 3rd Gen"],
    "vultr":        ["Cascade Lake", "EPYC 2nd Gen", "EPYC 3rd Gen"],
    "linode":       ["EPYC 1st Gen", "EPYC 2nd Gen", "EPYC 3rd Gen"],
    "leaseweb":     ["Cascade Lake", "Sapphire Rapids", "EPYC 2nd Gen", "EPYC 3rd Gen"],
    "clouvider":    ["Cascade Lake", "EPYC 3rd Gen"],
    "latitude":     ["EPYC 3rd Gen", "EPYC 4th Gen", "EPYC 5th Gen"],
    "oracle":       ["EPYC 3rd Gen", "EPYC 4th Gen", "EPYC 5th Gen"],
}

NON_HYPERSCALE_VCPU_MAX_W = {
    provider: round(
        sum(_CCF_MICROARCH_MAX_W[arch] for arch in archs) / len(archs),
        4,
    )
    for provider, archs in _PROVIDER_MICROARCHS.items()
}

_ALL_VCPU_MAX_W = {**CCF_VCPU_MAX_W, **NON_HYPERSCALE_VCPU_MAX_W}

if set(_ALL_VCPU_MAX_W) != set(PROVIDERS):
    raise ValueError(f"vCPU max power coverage mismatch: {set(PROVIDERS) ^ set(_ALL_VCPU_MAX_W)}")

In [ ]:
df["required_ram_gb"] = np.where(df["is_validator"], NODE_RAM_VALIDATOR_GB, NODE_RAM_NON_VALIDATOR_GB)
df["required_vcpu"] = np.where(df["is_validator"], NODE_VCPU_MIN_VALIDATOR, NODE_VCPU_MIN_NON_VALIDATOR)
df["required_ssd_tb"] = np.where(df["is_validator"], NODE_SSD_VALIDATOR_TB, NODE_SSD_NON_VALIDATOR_TB)

aws_mask = df["cloud_provider"] == "aws"
for idx, row in df[aws_mask].iterrows():
    node_arch = row["hw_arch"]
    candidates = {
        k: v for k, v in AWS_EC2_INSTANCE_POWER_W.items()
        if v["vcpu"] >= row["required_vcpu"]
        and v["ram_gb"] >= row["required_ram_gb"]
        and v["arch"] == node_arch
    }

    if not candidates:
        raise ValueError(f"No EC2 candidates found.")

    best = min(candidates.values(), key=lambda v: (v["vcpu"], v["ram_gb"]))
    p_cloud_load = best["pct100"]
    df.at[idx, "power_node_w"] = (p_cloud_load + SSD_OVERHEAD_W_PANKOVSKA) * CLOUD_PUE_PANKOVSKA

non_aws_cloud_mask = df["cloud_provider"].notna() & (df["cloud_provider"] != "aws")
for idx, row in df[non_aws_cloud_mask].iterrows():
    vcpu_min = row["required_vcpu"]
    p_cpu_w = _ALL_VCPU_MAX_W[row["cloud_provider"]] * vcpu_min
    p_ram_w = TEADS_MEMORY_W_PER_GB * row["required_ram_gb"]
    p_ssd_w = CCF_SSD_W_PER_TB * row["required_ssd_tb"]
    df.at[idx, "power_node_w"] = (p_cpu_w + p_ram_w + p_ssd_w) * CLOUD_PUE_CCF

print(df.shape)

## Model

In [ ]:
train_data = df.copy()

In [ ]:
assert train_data["consensus_client"].isna().sum() == 0, "unexpected NAs in consensus_client"
print(train_data["consensus_client"].value_counts(dropna=False).to_string())
print()

assert train_data["execution_client"].isna().sum() == 0, "unexpected NAs in execution_client"
print(train_data["execution_client"].value_counts(dropna=False).to_string())
print()

assert train_data["hw_arch"].isna().sum() == 0, "unexpected NAs in hw_arch"
print(train_data["hw_arch"].value_counts(dropna=False).to_string())
print()

assert train_data["os_token"].isna().sum() == 0, "unexpected NAs in os_token"
print(train_data["os_token"].value_counts(dropna=False).to_string())
print()

train_data["cloud_provider"] = train_data["cloud_provider"].fillna("bare_metal")
print(train_data["cloud_provider"].value_counts(dropna=False).to_string())
print()

assert train_data["attnets_num"].isna().sum() == 0, "unexpected NAs in attnets_num"
print(train_data["attnets_num"].value_counts(dropna=False).to_string())
print()

assert train_data["syncnets_num"].isna().sum() == 0, "unexpected NAs in syncnets_num"
print(train_data["syncnets_num"].value_counts(dropna=False).to_string())

In [ ]:
_KEEP_FOR_TRAIN_DATA = [
    "consensus_client",
    "execution_client",
    "hw_arch",
    "os_token",
    "cloud_provider",
    "attnets_num",
    "syncnets_num",
    "gossipsub_att_count",
    "power_node_w",
]

train_data = train_data[_KEEP_FOR_TRAIN_DATA]

print(train_data.shape)

In [ ]:
CATEGORICAL_FEATURES = [
    "consensus_client",
    "execution_client",
    "hw_arch",
    "os_token",
    "cloud_provider",
]
NUMERICAL_FEATURES = ["attnets_num", "syncnets_num", "gossipsub_att_count"]
TARGET = "power_node_w"

MODEL_PATH = "rf_power_model.joblib"

N_ESTIMATORS = 500
TEST_SIZE = 0.2
RANDOM_STATE = 42
N_CV_FOLDS = 5

preprocessor = ColumnTransformer(
    [(
        "cat",
        OrdinalEncoder(
            handle_unknown="use_encoded_value",
            unknown_value=-1,
            encoded_missing_value=-2,
        ),
        CATEGORICAL_FEATURES,
    )],
    remainder="passthrough",
)

model = RandomForestRegressor(
    n_estimators=N_ESTIMATORS,
    oob_score=True,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)
pipeline = Pipeline([("preprocess", preprocessor), ("model", model)])


def compute_mdi_importances(pipeline):
    raw_names = pipeline.named_steps["preprocess"].get_feature_names_out()
    stripped = [n.split("__")[-1] for n in raw_names]
    raw_imps = pipeline.named_steps["model"].feature_importances_
    grouped: dict = {}
    for name, imp in zip(stripped, raw_imps):
        grouped[name] = grouped.get(name, 0.0) + float(imp)
    return dict(sorted(grouped.items(), key=lambda kv: -kv[1]))


def compute_permutation_importances(pipeline, x, y):
    result = sk_permutation_importance(
        pipeline, x, y, n_repeats=10, random_state=RANDOM_STATE, n_jobs=-1
    )
    return dict(
        sorted(
            {
                feat: float(mean)
                for feat, mean in zip(list(x.columns), result.importances_mean)
            }.items(),
            key=lambda kv: -kv[1],
        )
    )


features = train_data[CATEGORICAL_FEATURES + NUMERICAL_FEATURES]
target = train_data[TARGET]

cv_scores = cross_val_score(
    pipeline, features, target, cv=N_CV_FOLDS, scoring="r2", n_jobs=-1
)

x_train, x_test, y_train, y_test = train_test_split(
    features, target, test_size=TEST_SIZE, random_state=RANDOM_STATE
)

pipeline.fit(x_train, y_train)

predictions = pipeline.predict(x_test)
metrics = {
    "n_train": len(x_train),
    "n_test": len(x_test),
    "rmse": mean_squared_error(y_test, predictions) ** 0.5,
    "mae": mean_absolute_error(y_test, predictions),
    "r2": r2_score(y_test, predictions),
    "oob_r2": pipeline.named_steps["model"].oob_score_,
    "cv_r2_mean": float(cv_scores.mean()),
    "cv_r2_std": float(cv_scores.std()),
    "y_test_mean": float(y_test.mean()),
    "y_test_std": float(y_test.std()),
    "mdi_importances": compute_mdi_importances(pipeline),
    "permutation_importances": compute_permutation_importances(pipeline, x_test, y_test),
}

print(json.dumps(metrics, indent=2))

joblib.dump(pipeline, MODEL_PATH)

In [ ]:
def split_recoverable_and_invalid(df_unfiltered):
    missing_required = (
        df_unfiltered["hw_arch"].isna()
        | df_unfiltered["consensus_client"].isna()
        | df_unfiltered["execution_client"].isna()
        | df_unfiltered["os_token"].isna()
    )
    caplin_invalid_mask = (
        (df_unfiltered["consensus_client"] == "caplin")
        & (df_unfiltered["execution_client"] != "erigon")
    )
    invalid_df = df_unfiltered[caplin_invalid_mask].copy()
    recoverable_df = df_unfiltered[missing_required & ~caplin_invalid_mask].copy()
    return recoverable_df, invalid_df


df_unfiltered = (
    df_merged[[c for c in _KEEP_FOR_INFERENCE]]
    .rename(columns=_KEEP_FOR_INFERENCE)
    .copy()
)

recoverable, invalid = split_recoverable_and_invalid(df_unfiltered)

recoverable["cloud_provider"] = recoverable["cloud_provider"].fillna("bare_metal")

remaining_cats = [c for c in CATEGORICAL_FEATURES if c != "cloud_provider"]
recoverable[remaining_cats] = recoverable[remaining_cats].fillna("unknown")

recoverable["power_node_w"] = pipeline.predict(
    recoverable[CATEGORICAL_FEATURES + NUMERICAL_FEATURES]
)
recoverable["power_source"] = "model_estimate"

labelled = df.copy()
labelled["power_source"] = "rule_based"

output_columns = CATEGORICAL_FEATURES + NUMERICAL_FEATURES + ["power_node_w", "power_source"]
extended = pd.concat(
    [labelled[output_columns], recoverable[output_columns]],
    ignore_index=True,
)

print(f"rule_based rows        : {len(labelled)}")
print(f"model_estimate rows     : {len(recoverable)}")
print(f"unrecoverable rows      : {len(invalid)}")
print(f"total matched peers     : {len(df_unfiltered)}")
print(f"coverage after extension: {len(extended)} / {len(df_unfiltered)}")

## Plots

### Plots about the model

In [ ]:
_DPI   = 150
_SERIF = "serif"
_ITEMS_DIR = Path(os.environ["ETH_ITEMS_DIR"]) if "ETH_ITEMS_DIR" in os.environ else None
_FIG_DIR = (_ITEMS_DIR / "figures") if _ITEMS_DIR else Path("figures")
_FIG_DIR.mkdir(parents=True, exist_ok=True)
 
_C_CL    = "#1c3d5a"
_C_EL    = "#245f7a"
_C_CLOUD = "#3d7eb5"
_C_BARE  = "#7f95b0"
_C_MEAN  = "#c0392b"
_C_MED   = "#e07b39"
_C_IMP   = "#1c3d5a"
 
plt.rcParams.update({
    "font.family":         _SERIF,
    "font.size":           9,
    "axes.linewidth":      0.6,
    "xtick.major.width":   0.5,
    "ytick.major.width":   0.5,
    "axes.spines.top":     False,
    "axes.spines.right":   False,
    "savefig.dpi":         _DPI,
    "savefig.bbox":        "tight",
    "figure.constrained_layout.use": True,
})
 
_PROVIDER_LABELS = {
    "bare_metal":    "Bare-metal",
    "hetzner":       "Hetzner",
    "ovh":           "OVH",
    "aws":           "AWS",
    "netcup":        "Netcup",
    "latitude":      "Latitude",
    "gcp":           "GCP",
    "oracle":        "Oracle",
    "digitalocean":  "DigitalOcean",
    "contabo":       "Contabo",
    "linode":        "Linode",
    "vultr":         "Vultr",
    "leaseweb":      "Leaseweb",
    "clouvider":     "Clouvider",
}
_CLOUD_SET = set(_PROVIDER_LABELS) - {"bare_metal"}
 
_df = df.copy()
_df["cloud_provider"] = _df["cloud_provider"].fillna("bare_metal")

In [ ]:
fig, (ax_cl, ax_el) = plt.subplots(1, 2, figsize=(11, 3.6))
 
for ax, col, title, color in [
    (ax_cl, "consensus_client", "(a)  Consensus Layer", _C_CL),
    (ax_el, "execution_client", "(b)  Execution Layer", _C_EL),
]:
    vc   = _df[col].value_counts().sort_values(ascending=True)
    pct  = vc / vc.sum() * 100
    bars = ax.barh(pct.index, pct.values, color=color, height=0.55, zorder=2)
    ax.set_xlabel("Share of labeled nodes (%)", labelpad=4)
    ax.set_title(title, fontweight="bold", pad=5)
    ax.set_xlim(0, pct.max() * 1.30)
    ax.tick_params(axis="y", length=0, pad=4)
    ax.grid(axis="x", linewidth=0.35, color="#cccccc", zorder=1)
    for bar, v in zip(bars, pct.values):
        ax.text(v + 0.4, bar.get_y() + bar.get_height() / 2,
                f"{v:.1f}%", va="center", fontsize=8)
 
plt.savefig(_FIG_DIR / "fig_client_distribution.png")
plt.show()

In [ ]:
prov_vc  = _df["cloud_provider"].value_counts().sort_values(ascending=True)
p_labels = [_PROVIDER_LABELS.get(k, k) for k in prov_vc.index]
p_colors = [_C_CLOUD if k in _CLOUD_SET else _C_BARE for k in prov_vc.index]

fig, ax_p = plt.subplots(figsize=(7, 4.8))
ax_p.barh(p_labels, prov_vc.values, color=p_colors, height=0.6, zorder=2)
ax_p.set_xlabel("Number of labeled nodes", labelpad=4)
ax_p.set_title("Node count by hosting provider", fontweight="bold", pad=5)
ax_p.tick_params(axis="y", length=0, pad=4)
ax_p.grid(axis="x", linewidth=0.35, color="#cccccc", zorder=1)
_prov_total = prov_vc.sum()
for i, (v, k) in enumerate(zip(prov_vc.values, prov_vc.index)):
    pct = v / _prov_total * 100
    ax_p.text(v + max(prov_vc.values) * 0.01, i, f"{v} ({pct:.1f}%)", va="center", fontsize=7.5)
ax_p.set_xlim(0, max(prov_vc.values) * 1.30)
ax_p.legend(
    handles=[
        mpatches.Patch(color=_C_CLOUD, label="Cloud"),
        mpatches.Patch(color=_C_BARE,  label="Bare-metal"),
    ],
    fontsize=8, framealpha=0.75, loc="lower right",
)

plt.savefig(_FIG_DIR / "fig_provider_distribution.png")
plt.show()

In [ ]:
goss = _df["gossipsub_att_count"].astype(int)
goss_max = max(goss.max(), 64)
_goss_bins = range(0, goss_max + 2)

fig, ax_g = plt.subplots(figsize=(6.5, 4.8))
_goss_counts, _goss_edges, _goss_patches = ax_g.hist(
    goss, bins=_goss_bins, color=_C_CL, edgecolor="white", linewidth=0.5, zorder=2
)
ax_g.set_yscale("log")

_counts_series = pd.Series(_goss_counts, index=range(len(_goss_counts)))
_top_vals = _counts_series[_counts_series > 0].nlargest(4).index.tolist()
_top_vals_sorted = sorted(_top_vals)
_n_total = len(goss)

_HIGHLIGHT_COLORS = ["#e07b39", "#c0392b", "#2ecc71", "#8e44ad"]
_val_to_color = {v: _HIGHLIGHT_COLORS[i] for i, v in enumerate(_top_vals_sorted)}

for _v, _color in _val_to_color.items():
    _goss_patches[_v].set_facecolor(_color)

from matplotlib.lines import Line2D
_legend_handles = [
    Line2D([], [], linestyle="none", marker="s", color=_val_to_color[_v], markersize=7,
           label=f"gossipsub_att_count = {_v}:  {int(_goss_counts[_v]):,} nodes  ({int(_goss_counts[_v]) / _n_total * 100:.1f}%)")
    for _v in _top_vals_sorted
]
ax_g.legend(
    handles=_legend_handles,
    fontsize=7.5, framealpha=0.92, edgecolor="#cccccc",
    loc="upper right",
    prop={"family": "monospace", "size": 7.5},
)

ax_g.set_xlabel("Gossipsub attestation topic subscriptions", labelpad=4)
ax_g.set_ylabel("Number of nodes", labelpad=4)
ax_g.set_title("Gossipsub attestation topic distribution (gossipsub_att_count)", fontweight="bold", pad=5)
ax_g.set_xlim(-0.5, goss_max + 0.5)
ax_g.grid(axis="y", linewidth=0.35, color="#cccccc", zorder=1)
ax_g.tick_params(axis="x", length=2)

plt.savefig(_FIG_DIR / "fig_gossipsub_distribution.png")
plt.show()

In [ ]:
from matplotlib.lines import Line2D
from matplotlib.ticker import MultipleLocator

pw     = _df["power_node_w"].dropna()
p_min  = pw.min()
p_q1   = pw.quantile(0.25)
p_med  = pw.median()
p_mean = pw.mean()
p_q3   = pw.quantile(0.75)
p_max  = pw.max()
p_std  = pw.std()
n      = len(pw)

fig, ax = plt.subplots(figsize=(10, 5))

_lo   = np.floor(p_min / 2) * 2
_hi   = np.ceil(p_max / 2) * 2
_bins = np.arange(_lo, _hi + 2.5, 2.5)

_counts, _, _patches = ax.hist(pw, bins=_bins, color=_C_CL, edgecolor="white",
                                linewidth=0.4, alpha=0.92, zorder=2)
ax.bar_label(_patches, labels=[f"{int(c)}" if c > 0 else "" for c in _counts],
             fontsize=6.5, padding=1, rotation=0)
ax.axvline(p_mean, color=_C_MEAN, linewidth=1.6, linestyle="--", zorder=4)
ax.axvline(p_med,  color=_C_MED,  linewidth=1.6, linestyle=(0, (1, 1)), zorder=4)

_blank = lambda: Line2D([], [], linestyle="none")
_row   = lambda k, v: f"{k:<7}= {v:>8}"
handles = [
    (_blank(),                                                    _row("n",      f"{n:,}")),
    (_blank(),                                                    _row("Min",    f"{p_min:.1f} W")),
    (_blank(),                                                    _row("Q1",     f"{p_q1:.1f} W")),
    (Line2D([], [], color=_C_MED,  linewidth=1.6, linestyle=(0, (1, 1))), _row("Median", f"{p_med:.1f} W")),
    (Line2D([], [], color=_C_MEAN, linewidth=1.6, linestyle="--"),        _row("Mean",   f"{p_mean:.1f} W")),
    (_blank(),                                                    _row("Q3",     f"{p_q3:.1f} W")),
    (_blank(),                                                    _row("Max",    f"{p_max:.1f} W")),
    (_blank(),                                                    _row("Std",    f"{p_std:.1f} W")),
]
ax.legend(
    [h for h, _ in handles], [t for _, t in handles],
    loc="upper left", fontsize=8.5, framealpha=0.92,
    edgecolor="#cccccc", handlelength=1.6, handletextpad=0.6,
    labelspacing=0.5, borderpad=0.7,
    prop={"family": "monospace", "size": 8.5},
)

ax.set_title(
    f"Power distribution",
    fontweight="bold", pad=8,
)
ax.set_xlabel("Estimated node power (W)", labelpad=4)
ax.set_ylabel("Quantity of nodes", labelpad=4)
ax.set_ylim(top=ax.get_ylim()[1] * 1.15)
ax.xaxis.set_major_locator(MultipleLocator(4))
ax.grid(axis="y", which="major", linewidth=0.35, color="#dddddd", zorder=1)
ax.margins(x=0.01)

plt.savefig(_FIG_DIR / "fig_power_distribution.png")
plt.show()


In [ ]:
fig, (ax_mdi, ax_perm) = plt.subplots(1, 2, figsize=(12, 3.8))
 
for ax, key, title, unit in [
    (ax_mdi,  "mdi_importances",          "(a)  MDI importance",
     "Fraction of total impurity reduction"),
    (ax_perm, "permutation_importances",   "(b)  Permutation importance",
     "Mean decrease in R² on test set"),
]:
    d     = metrics[key]
    feats = list(d.keys())[::-1]
    vals  = [d[f] for f in feats]
    lbls  = feats
    ax.barh(lbls, vals, color=_C_IMP, height=0.55, zorder=2)
    ax.set_title(title, fontweight="bold", pad=5)
    ax.set_xlabel(unit, labelpad=4)
    ax.tick_params(axis="y", length=0, pad=4)
    ax.grid(axis="x", linewidth=0.35, color="#cccccc", zorder=1)
    mx = max(vals)
    for i, v in enumerate(vals):
        fmt = f"{v:.3f}" if v >= 0.001 else f"{v:.4f}"
        ax.text(v + mx * 0.01, i, fmt, va="center", fontsize=7.5)
    ax.set_xlim(0, mx * 1.25)
 
plt.savefig(_FIG_DIR / "fig_feature_importances.png")
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(5, 5))

ax.scatter(y_test, predictions, alpha=0.4, s=14, color=_C_CL, zorder=2)

lims = [
    min(float(y_test.min()), float(predictions.min())) - 1,
    max(float(y_test.max()), float(predictions.max())) + 1,
]
ax.plot(lims, lims, "--", color=_C_MEAN, linewidth=1.2, label="Identity (perfect fit)")
ax.set_xlim(lims)
ax.set_ylim(lims)

ax.set_xlabel("True power (W)", labelpad=4)
ax.set_ylabel("Predicted power (W)", labelpad=4)
ax.set_title(
    rf"Predicted vs.\ actual --- hold-out test set ($n = {metrics['n_test']}$)",
    fontweight="bold", pad=6,
)
ax.legend(fontsize=8, framealpha=0.85)
ax.grid(linewidth=0.35, color="#cccccc", zorder=1)

plt.savefig(_FIG_DIR / "fig_pred_vs_actual.png")
plt.show()

### Plots about the empirical measurements

In [ ]:
_NIMBUS_GETH_CSV      = "reports_nimbus_geth.csv"
_PRYSM_NETHERMIND_CSV = "reports_prysm_nethermind.csv"
_IDLE_CSV             = "reports_idle.csv"

_RASPBERRY_CSV             = "reports_raspberry_helios.csv"

_RAW_POWER_COL = "power (Watt)"
_PLOT_DATE_NIM_GETH     = "2026-07-11"
_PLOT_PRYSM_NETHER_START = "2026-07-20 14:00"
_PLOT_PRYSM_NETHER_END   = "2026-07-21 14:00"
_PLOT_IDLE_START           = "2026-07-16 18:00"
_PLOT_IDLE_END             = "2026-07-17 18:00"

_PLOT_RUNNING_RASPBERRY_START           = "2026-07-30 01:20"
_PLOT_RUNNING_RASPBERRY_END             = "2026-07-30 07:20"

_PLOT_IDLE_RASPBERRY_START           = "2026-07-27 08:30"
_PLOT_IDLE_RASPBERRY_END             = "2026-07-27 14:30"

_SMOOTH_WINDOW = 20

def _load_power_csv(filename):
    path = hf_hub_download(repo_id=HF_REPO_ID, repo_type=HF_REPO_TYPE, filename=filename)
    df = pd.read_csv(path, parse_dates=["time"])
    df = df.rename(columns={_RAW_POWER_COL: "power"})[["time", "power"]]
    return df.sort_values("time").reset_index(drop=True)

def _filter_day(df, date_str):
    return df[df["time"].dt.date.astype(str) == date_str].copy()

In [ ]:
df_ng_full = _load_power_csv(_NIMBUS_GETH_CSV)
df_ng_day  = _filter_day(df_ng_full, _PLOT_DATE_NIM_GETH)

df_pn_full   = _load_power_csv(_PRYSM_NETHERMIND_CSV)
df_idle_full = _load_power_csv(_IDLE_CSV)

df_pn_day = df_pn_full[
    (df_pn_full["time"] >= pd.Timestamp(_PLOT_PRYSM_NETHER_START))
    & (df_pn_full["time"] <= pd.Timestamp(_PLOT_PRYSM_NETHER_END))
].copy()

df_idle_day = df_idle_full[
    (df_idle_full["time"] >= pd.Timestamp(_PLOT_IDLE_START))
    & (df_idle_full["time"] <= pd.Timestamp(_PLOT_IDLE_END))
].copy()

In [ ]:
PC_HW_ARCH  = "x86"
PC_OS_TOKEN = "windows"
PC_ATTNETS  = 2
PC_SYNCNETS = 0
PC_GOSSIBSUB_ATT_COUNT = 2
PC_PROVIDER = "bare_metal"

def _predict_pc_combo(cl, el):
    row = pd.DataFrame([{
        "consensus_client": cl,
        "execution_client": el,
        "hw_arch":          PC_HW_ARCH,
        "os_token":         PC_OS_TOKEN,
        "cloud_provider":   PC_PROVIDER,
        "attnets_num":      PC_ATTNETS,
        "syncnets_num":     PC_SYNCNETS,
        "gossipsub_att_count": PC_GOSSIBSUB_ATT_COUNT,
    }])
    return float(pipeline.predict(row)[0])

pred_ng = _predict_pc_combo("nimbus", "geth")
pred_pn = _predict_pc_combo("prysm", "nethermind")


In [ ]:
P_GPU_IDLE = 33.0

_C_NG_PLOT   = "#f2a154"
_C_PN_PLOT   = "#5b9bd5"
_C_IDLE_PLOT = "#8a97a8"
_C_PRED_PLOT = "#8e24aa"

def _plot_combo(ax, df_day, pred, gpu_adjusted, color, series_label):
    power_series = df_day["power"]
    idle_series  = df_idle_day["power"]
    if gpu_adjusted:
        power_series = (power_series - P_GPU_IDLE).clip(lower=0)
        idle_series  = (idle_series - P_GPU_IDLE).clip(lower=0)

    elapsed_h    = (df_day["time"]      - df_day["time"].iloc[0]).dt.total_seconds()      / 3600
    smooth       = power_series.rolling(_SMOOTH_WINDOW, center=True, min_periods=1).mean()
    idle_elapsed = (df_idle_day["time"] - df_idle_day["time"].iloc[0]).dt.total_seconds()  / 3600
    idle_smooth  = idle_series.rolling(_SMOOTH_WINDOW, center=True, min_periods=1).mean()

    ax.plot(idle_elapsed, idle_smooth, color=_C_IDLE_PLOT, linewidth=1.4,
            label="Measured idle", zorder=2)
    ax.plot(elapsed_h, smooth, color=color, linewidth=2.0, label=series_label, zorder=3)
    ax.axhline(pred, color=_C_PRED_PLOT, linewidth=2.4, linestyle="--", zorder=4, label="Model prediction")

    ax.annotate(f"{pred:.1f} W", xy=(1, pred), xycoords=("axes fraction", "data"),
                xytext=(-8, 0), textcoords="offset points",
                ha="right", va="center", fontsize=14, color=_C_PRED_PLOT,
                fontweight="bold", zorder=5,
                bbox=dict(boxstyle="round,pad=0.25", facecolor="white",
                          edgecolor=_C_PRED_PLOT, linewidth=1.4, alpha=0.95))

    ax.set_xlim(0, 24)
    ax.set_xticks(range(0, 25, 4))
    ax.set_xticklabels([f"{h}h" for h in range(0, 25, 4)])
    ax.set_xlabel("Elapsed time in 24 h window", labelpad=6, fontsize=14)
    ax.set_ylabel("Power (W)", fontsize=14, labelpad=6)
    ax.tick_params(axis="both", labelsize=12.5)
    ax.spines[["top", "right"]].set_visible(False)


def _make_row_figure(gpu_adjusted, suptitle, filename):
    fig, (ax_left, ax_right) = plt.subplots(1, 2, figsize=(15, 5))
    fig.subplots_adjust(wspace=0.32)

    _plot_combo(ax_left,  df_ng_day, pred_ng, gpu_adjusted, _C_NG_PLOT, "(Measured) Nimbus + Geth")
    _plot_combo(ax_right, df_pn_day, pred_pn, gpu_adjusted, _C_PN_PLOT, "(Measured) Prysm + Nethermind")

    _handles, _labels = [], []
    for ax in (ax_left, ax_right):
        for h, l in zip(*ax.get_legend_handles_labels()):
            if l not in _labels:
                _handles.append(h)
                _labels.append(l)

    fig.legend(_handles, _labels, loc="upper center", ncol=len(_labels),
               bbox_to_anchor=(0.5, 1.14), fontsize=14, frameon=False)

    fig.suptitle(suptitle, fontsize=17, fontweight="bold", y=1.22)

    plt.savefig(_FIG_DIR / filename, bbox_inches="tight")
    plt.show()


_make_row_figure(False, "24-hour power traces: measured vs. rule-based prediction \u2014 Raw measured",
                  "fig_empirical_comparison_raw.png")
_make_row_figure(True, "24-hour power traces: measured vs. rule-based prediction \u2014 GPU-adjusted",
                  "fig_empirical_comparison_gpu_adjusted.png")

In [ ]:
df_rpi_full = _load_power_csv(_RASPBERRY_CSV)

df_rpi_running = df_rpi_full[
    (df_rpi_full["time"] >= pd.Timestamp(_PLOT_RUNNING_RASPBERRY_START))
    & (df_rpi_full["time"] <= pd.Timestamp(_PLOT_RUNNING_RASPBERRY_END))
].copy()

df_rpi_idle = df_rpi_full[
    (df_rpi_full["time"] >= pd.Timestamp(_PLOT_IDLE_RASPBERRY_START))
    & (df_rpi_full["time"] <= pd.Timestamp(_PLOT_IDLE_RASPBERRY_END))
].copy()

In [ ]:
_C_HELIOS_PLOT = "#3cb371"
_C_RPI_MEAN    = _C_MEAN

fig, (ax_idle, ax_run) = plt.subplots(1, 2, figsize=(14, 5))
fig.subplots_adjust(wspace=0.28)

elapsed_idle = (df_rpi_idle["time"]    - df_rpi_idle["time"].iloc[0]).dt.total_seconds()    / 3600
elapsed_run  = (df_rpi_running["time"] - df_rpi_running["time"].iloc[0]).dt.total_seconds() / 3600

smooth_idle = df_rpi_idle["power"].rolling(_SMOOTH_WINDOW, center=True, min_periods=1).mean()
smooth_run  = df_rpi_running["power"].rolling(_SMOOTH_WINDOW, center=True, min_periods=1).mean()

mean_idle = df_rpi_idle["power"].mean()
mean_run  = df_rpi_running["power"].mean()

for ax, elapsed, smooth, mean_val, color, title in [
    (ax_idle, elapsed_idle, smooth_idle, mean_idle, _C_IDLE_PLOT,   "(a)  Idle"),
    (ax_run,  elapsed_run,  smooth_run,  mean_run,  _C_HELIOS_PLOT, "(b)  Helios light client"),
]:
    ax.plot(elapsed, smooth, color=color, linewidth=2.0, zorder=3)
    ax.axhline(mean_val, color=_C_RPI_MEAN, linewidth=1.4, linestyle="--", zorder=4, alpha=0.85,
               label=f"Mean: {mean_val:.2f} W")
    leg = ax.legend(loc="lower right", fontsize=12, frameon=True,
                    facecolor="white", edgecolor="#cccccc", framealpha=0.95)
    ax.set_xlim(0, 6)
    ax.set_xticks(range(0, 7))
    ax.set_xticklabels([f"{h}h" for h in range(0, 7)])
    ax.set_xlabel("Elapsed time in 6 h window", labelpad=6, fontsize=14)
    ax.set_ylabel("Power (W)", fontsize=14, labelpad=6)
    ax.tick_params(axis="both", labelsize=12.5)
    ax.spines[["top", "right"]].set_visible(False)
    ax.set_title(title, fontsize=14, fontweight="bold", pad=8)

fig.suptitle("Raspberry Pi Zero 2W — idle vs. Helios light client", fontsize=15, fontweight="bold", y=1.10)

plt.savefig(_FIG_DIR / "fig_raspberry_helios.png", bbox_inches="tight", dpi=_DPI)
plt.show()

## Tables

In [ ]:
_TAB_DIR = (_ITEMS_DIR / "tables") if _ITEMS_DIR else Path("tables")
_TAB_DIR.mkdir(parents=True, exist_ok=True)

def _fmt_int(n):
    return f"{n:,}".replace(",", "{,}")

In [ ]:
n_total = len(df_unfiltered)
n_labelled = len(labelled)
n_recoverable = len(recoverable)
n_invalid = len(invalid)

tab_coverage = rf"""\begin{{tabular}}{{lr}}
\toprule
Category & Count \\
\midrule
Total CL$+$EL inner-joined peers  & {_fmt_int(n_total)} \\
Rule-based labeled & {_fmt_int(n_labelled)} \\
Recoverable via predictive model   &  {_fmt_int(n_recoverable)} \\
Invalid     &      {_fmt_int(n_invalid)} \\
\bottomrule
\end{{tabular}}
"""

(_TAB_DIR / "tab_coverage.tex").write_text(tab_coverage)

In [ ]:
normalised_mae_pct = metrics["mae"] / metrics["y_test_mean"] * 100

tab_model_metrics = rf"""\begin{{tabular}}{{lr}}
\toprule
Metric & Value \\
\midrule
Training set size          & {_fmt_int(metrics["n_train"])} nodes \\
Test set size              & {_fmt_int(metrics["n_test"])} nodes \\
Hold-out RMSE              & {metrics["rmse"]:.2f}~W \\
Hold-out MAE               & {metrics["mae"]:.2f}~W \\
Normalised MAE             & {normalised_mae_pct:.2f}\% \\
Hold-out $R^2$             & {metrics["r2"]:.4f} \\
OOB $R^2$                  & {metrics["oob_r2"]:.4f} \\
CV $R^2$ mean (5-fold)     & {metrics["cv_r2_mean"]:.4f} \\
CV $R^2$ std (5-fold)      & {metrics["cv_r2_std"]:.4f} \\
$\bar{{y}}_{{\text{{test}}}}$    & {metrics["y_test_mean"]:.1f}~W \\
$\sigma_{{\text{{test}}}}$     & {metrics["y_test_std"]:.1f}~W \\
\bottomrule
\end{{tabular}}
"""

(_TAB_DIR / "tab_model_metrics.tex").write_text(tab_model_metrics)

In [ ]:
p_meas_ng = df_ng_day["power"].mean()
p_meas_pn = df_pn_day["power"].mean()

delta_ng = abs(p_meas_ng - pred_ng)
delta_pn = abs(p_meas_pn - pred_pn)

rel_ng = delta_ng / p_meas_ng * 100
rel_pn = delta_pn / p_meas_pn * 100

p_meas_ng_adj = (df_ng_day["power"] - P_GPU_IDLE).clip(lower=0).mean()
p_meas_pn_adj = (df_pn_day["power"] - P_GPU_IDLE).clip(lower=0).mean()

delta_ng_adj = abs(p_meas_ng_adj - pred_ng)
delta_pn_adj = abs(p_meas_pn_adj - pred_pn)

rel_ng_adj = delta_ng_adj / p_meas_ng_adj * 100
rel_pn_adj = delta_pn_adj / p_meas_pn_adj * 100

tab_empirical_comparison = rf"""\renewcommand{{\arraystretch}}{{1.3}}
\setlength{{\tabcolsep}}{{14pt}}
\resizebox{{\textwidth}}{{!}}{{%
\begin{{tabular}}{{l !{{\color{{gray!40}}\vrule width 0.4pt}} c !{{\color{{gray!40}}\vrule width 0.4pt}} r !{{\color{{gray!40}}\vrule width 0.4pt}} r !{{\color{{gray!40}}\vrule width 0.4pt}} r}}
\toprule
Combination
  & GPU-adjustment$^{{*}}$
  & $P_{{\mathrm{{measured}}}}$ (24-hr mean) [W]
  & $P_{{\mathrm{{predicted}}}}$ [W]
  & $|\Delta P| / P_{{\mathrm{{measured}}}}$ [\%] \\
\midrule
Nimbus + Geth     & False & {p_meas_ng:.2f} & {pred_ng:.1f} & {rel_ng:.1f} \\
Prysm + Nethermind & False & {p_meas_pn:.2f} & {pred_pn:.1f} & {rel_pn:.1f} \\
Nimbus + Geth      & True & {p_meas_ng_adj:.2f} & {pred_ng:.1f} & {rel_ng_adj:.1f} \\
Prysm + Nethermind & True & {p_meas_pn_adj:.2f} & {pred_pn:.1f} & {rel_pn_adj:.1f} \\
\bottomrule
\end{{tabular}}%
}}
\renewcommand{{\arraystretch}}{{1}}
\vspace{{2pt}}

{{\footnotesize $^{{*}}$the GPU-adjustment here consists of the subtraction of the RTX~4090 idle draw of 33~W (\textcite{{servethehome_rtx4090}}, RTX~4090 Founders Edition review) from the measured 24-hour mean.}}
"""

(_TAB_DIR / "tab_empirical_comparison.tex").write_text(tab_empirical_comparison)


In [ ]:
_ORDERED_EC2 = {
    **_M6I_INSTANCES,
    **_R6I_INSTANCES,
    **_R6G_INSTANCES,
    **_M7G_INSTANCES,
}

rows = "\n".join(
    rf"{name} & {v['arch']} & {v['vcpu']} & {v['ram_gb']}"
    rf" & {v['pkg_w_100']:.2f} & {v['ram_w_100']:.2f} & {v['delta']:.2f} & {v['pct100']:.2f} \\"
    for name, v in _ORDERED_EC2.items()
)

tab_ec2_instances = rf"""\begin{{tabular}}{{llrrrrrr}}
\toprule
Instance & Arch & \ac{{vcpu}} & \ac{{ram}} [GB] & $P_{{\text{{pkg}}}}$ [W] & $P_{{\text{{RAM}}}}$ [W] & $\delta$ [W] & $P_{{\text{{pct100}}}}$ [W] \\
\midrule
{rows}
\bottomrule
\end{{tabular}}
"""

(_TAB_DIR / "tab_ec2_instances.tex").write_text(tab_ec2_instances)


In [ ]:
att_vc = _df["attnets_num"].astype(int).value_counts().sort_index()
att_total = att_vc.sum()

rows = "\n".join(
    rf"{v} & {_fmt_int(c)} & {c / att_total * 100:.1f}\% \\"
    for v, c in att_vc.items()
)

tab_attnets_distribution = rf"""\begin{{tabular}}{{rrr}}
\toprule
\texttt{{attnets\_num}} & Nodes & Share [\%] \\
\midrule
{rows}
\midrule
Total & {_fmt_int(att_total)} & 100.0\% \\
\bottomrule
\end{{tabular}}
"""

(_TAB_DIR / "tab_attnets_distribution.tex").write_text(tab_attnets_distribution)

In [ ]:
sync_vc = _df["syncnets_num"].astype(int).value_counts().sort_index()
sync_total = sync_vc.sum()

rows = "\n".join(
    rf"{v} & {_fmt_int(c)} & {c / sync_total * 100:.1f}\% \\"
    for v, c in sync_vc.items()
)

tab_syncnets_distribution = rf"""\begin{{tabular}}{{rrr}}
\toprule
\texttt{{syncnets\_num}} & Nodes & Share [\%] \\
\midrule
{rows}
\midrule
Total & {_fmt_int(sync_total)} & 100.0\% \\
\bottomrule
\end{{tabular}}
"""

(_TAB_DIR / "tab_syncnets_distribution.tex").write_text(tab_syncnets_distribution)